In [ ]:
!pip install -q numpy==1.26.4
!pip install -q faiss-cpu sentence-transformers transformers accelerate bitsandbytes peft sentencepiece scikit-learn pandas rouge-score nltk fastapi uvicorn pyngrok nest-asyncio
print("✅ Kurulum tamam.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 98.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have nu

In [ ]:
import os, json, math, re, time, warnings, zipfile
import pandas as pd
import torch


warnings.filterwarnings("ignore")

from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("Tüm importlar tamam.")
print("CUDA:", torch.cuda.is_available())

Tüm importlar tamam.
CUDA: True


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


FAISS_INDEX_PATH = "/content/drive/MyDrive/endodonti_rag/outputs/faiss/BGE_Large/index.faiss"
CHUNKS_PATH      = "/content/chunks_temiz (1).jsonl"
ZIP_PATH         = "/content/drive/MyDrive/endodonti_final_paket.zip"
ADAPTER_PATH     = "/content/adapter"
OUTPUT_DIR       = "/content/drive/MyDrive/endodonti_rag/outputs"
EMBEDDING_MODEL  = "BAAI/bge-large-en-v1.5"
K_RETRIEVE       = 5

os.makedirs(ADAPTER_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR,   exist_ok=True)

print("index.faiss var mı:", os.path.exists(FAISS_INDEX_PATH))
print("chunks.jsonl var mı:", os.path.exists(CHUNKS_PATH))
print("adapter zip var mı:", os.path.exists(ZIP_PATH))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
index.faiss var mı: True
chunks.jsonl var mı: True
adapter zip var mı: True


In [ ]:
chunks = []
if os.path.exists(CHUNKS_PATH):
    with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            try:
                chunks.append(json.loads(line))
            except:
                pass
    print(f"✅ {len(chunks)} chunk yüklendi.")
else:
    print("⚠️ chunks.jsonl bulunamadı. Metadata olmadan devam edilecek.")

✅ 2046 chunk yüklendi.


In [ ]:
import faiss
print("faiss:", faiss.__version__)

faiss: 1.14.3


In [ ]:
embedding_device = "cuda" if torch.cuda.is_available() else "cpu"

emb_model = SentenceTransformer(EMBEDDING_MODEL, device=embedding_device)
index     = faiss.read_index(FAISS_INDEX_PATH)

print(f"✅ FAISS index yüklendi. Toplam vektör: {index.ntotal}")
print(f"✅ Embedding modeli hazır ({embedding_device})")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ FAISS index yüklendi. Toplam vektör: 2046
✅ Embedding modeli hazır (cuda)


In [ ]:
kaliteli_indexler = set()

for i, chunk in enumerate(chunks):
    metin = chunk.get("text", "")
    if "FIG." in metin or "Fig." in metin:
        continue
    if any(x in metin for x in ["J Endod", "J Am Dent", "Oral Surg",
                                  "Dent Mater", "Int Endod", ". 24.", ". 25.",
                                  "et al:", "et al."]):
        continue
    if len(metin) < 200:
        continue
    kaliteli_indexler.add(i)

print(f"✅ Kaliteli chunk: {len(kaliteli_indexler)} / {len(chunks)}")

✅ Kaliteli chunk: 1358 / 2046


In [ ]:
def similarity_search(query: str, k: int = K_RETRIEVE) -> list:
    vec = emb_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(vec, k * 5)

    results = []
    for rank, idx in enumerate(indices[0]):
        if idx == -1:
            continue


        if idx not in kaliteli_indexler:
            continue

        if idx < len(chunks):
            chunk = chunks[idx]
            text  = chunk.get("text", "")
            page  = chunk.get("page", chunk.get("page_number", "?"))
        else:
            text = ""
            page = "?"

        results.append({
            "page_content": text,
            "metadata":     {"page": page, "score": float(scores[0][rank])},
        })

        if len(results) == k:
            break

    return results


test_docs = similarity_search("EDTA endodontics smear layer", k=2)
for doc in test_docs:
    print(f"Sayfa {doc['metadata']['page']}: {doc['page_content'][:200]}")
    print()

Sayfa 1087: portion of the root, 183 including accessory RCS, areas of resorption and repaired resorptions, pulp stones, irregular or absent primary tubules, irregular secondary dentin, and cementum-like tissue l

Sayfa 3322: Epithelial stem cells, 477 Equivalent dose, 426 Er:YAG laser, 573–574 , 574f Esthetic bleaching, e79 Ethylenediamine tetra-acetic acid (EDTA) description of, 278–279 endodontic applications of, 279 , 



In [ ]:
if not os.path.exists(os.path.join(ADAPTER_PATH, "adapter_config.json")):
    print("Adapter zip'ten çıkarılıyor...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(ADAPTER_PATH)
    print("✅ Adapter çıkarıldı:", os.listdir(ADAPTER_PATH)[:5])
else:
    print("✅ Adapter zaten mevcut.")

with open(os.path.join(ADAPTER_PATH, "adapter_config.json"), "r") as f:
    adapter_cfg = json.load(f)

BASE_MODEL_ID = adapter_cfg.get("base_model_name_or_path", "mistralai/Mistral-7B-Instruct-v0.2")
print("Base model:", BASE_MODEL_ID)

Adapter zip'ten çıkarılıyor...
✅ Adapter çıkarıldı: ['adapter_model.safetensors', 'checkpoint-240', 'checkpoint-96', 'tokenizer_config.json', 'adapter_config.json']
Base model: mistralai/Mistral-7B-Instruct-v0.2


In [ ]:
import json, inspect
from peft import LoraConfig

config_path = "/content/adapter/adapter_config.json"

with open(config_path, "r") as f:
    cfg = json.load(f)


gecerli = set(inspect.signature(LoraConfig.__init__).parameters.keys())
gecerli.discard("self")

kaldirilanlar = {k: v for k, v in cfg.items() if k not in gecerli and k != "peft_type"}
temiz_cfg     = {k: v for k, v in cfg.items() if k in gecerli or k == "peft_type"}

print("Kaldırılan parametreler:", list(kaldirilanlar.keys()))

with open(config_path, "w") as f:
    json.dump(temiz_cfg, f, indent=2)

print("✅ Temizlendi. Kalan:", list(temiz_cfg.keys()))

Kaldırılan parametreler: []
✅ Temizlendi. Kalan: ['alora_invocation_tokens', 'alpha_pattern', 'arrow_config', 'auto_mapping', 'base_model_name_or_path', 'bias', 'corda_config', 'ensure_weight_tying', 'eva_config', 'exclude_modules', 'fan_in_fan_out', 'inference_mode', 'init_lora_weights', 'layer_replication', 'layers_pattern', 'layers_to_transform', 'loftq_config', 'lora_alpha', 'lora_bias', 'lora_dropout', 'lora_ga_config', 'megatron_config', 'megatron_core', 'modules_to_save', 'peft_type', 'peft_version', 'qalora_group_size', 'r', 'rank_pattern', 'revision', 'target_modules', 'target_parameters', 'task_type', 'trainable_token_indices', 'use_bdlora', 'use_dora', 'use_qalora', 'use_rslora']


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("GPU bulunamadı.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("⏳ Base model yükleniyor...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("⏳ LoRA adapter yükleniyor...")
expert_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
expert_model.eval()
print("🚀 Model hazır.")

⏳ Base model yükleniyor...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

⏳ LoRA adapter yükleniyor...
🚀 Model hazır.


In [ ]:
ALPACA_TMPL = (
    "### Instruction:\n"
    "Answer as an endodontist using only the given context. "
    "Give a concise answer in 1-2 sentences maximum. "
    "Preserve exact clinical and endodontic keywords from the context whenever possible. "
    "Do not replace technical terms with general language. "
    "Use abbreviations such as NaOCl, EDTA, MTA exactly if they appear in the context.\n\n"
    "### Input:\n"
    "CONTEXT:\n{context}\n\n"
    "Question: {question}\n\n"
    "### Response:\n"
)

def build_context(docs, max_chars=2200):
    parts, total = [], 0
    for i, doc in enumerate(docs, 1):
        text = (doc["page_content"] or "").strip()
        if not text:
            continue
        page    = doc["metadata"].get("page", "?")
        labeled = f"[Belge {i} | Sayfa {page}]\n{text}"
        if total + len(labeled) > max_chars:
            break
        parts.append(labeled)
        total += len(labeled)
    return "\n\n".join(parts)

def clean_answer(text: str) -> str:
    if "### Response:" in text:
        text = text.split("### Response:", 1)[-1]
    for marker in ["\nQuestion:", "\nAnswer:", "\n###"]:
        if marker in text:
            text = text.split(marker)[0]
    return text.strip()

def endodonti_asistanina_sor(question: str, show_context=False) -> str:
    docs    = similarity_search(question, k=K_RETRIEVE)
    context = build_context(docs)

    if show_context:
        print(context[:1200])

    prompt = ALPACA_TMPL.format(context=context, question=question)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536)
    inputs = {k: v.to(expert_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = expert_model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return clean_answer(decoded)

print("✅ RAG fonksiyonları hazır.")

✅ RAG fonksiyonları hazır.


In [ ]:
TEST_SORULARI = [
    "What is the main purpose of EDTA in endodontics?",
    "What are the primary objectives of using Sodium Hypochlorite (NaOCl)?",
    "What are the clinical signs of irreversible pulpitis?",
    "How should a clinician manage a separated instrument in the root canal?",
    "Explain the importance of working length determination in endodontics.",
]

sonuclar = []
for i, soru in enumerate(TEST_SORULARI, 1):
    t0 = time.time()
    print(f"{i}/{len(TEST_SORULARI)} cevaplanıyor: {soru}")
    try:
        cevap = endodonti_asistanina_sor(soru)
    except Exception as exc:
        cevap = f"[HATA] {exc}"
    sonuclar.append({
        "Soru_No":        i,
        "Soru":           soru,
        "Asistan_Cevabi": cevap,
        "Sure_sn":        round(time.time() - t0, 2),
    })
    print(f"✅ {round(time.time()-t0,1)}sn | {cevap[:200]}\n")

df_final = pd.DataFrame(sonuclar)
df_final

1/5 cevaplanıyor: What is the main purpose of EDTA in endodontics?
✅ 11.3sn | EDTA is used as a canal irrigant in endodontics due to its ability to selectively remove mineral from a dentin surface, exposing a collagenous matrix. It is

2/5 cevaplanıyor: What are the primary objectives of using Sodium Hypochlorite (NaOCl)?
✅ 13.7sn | Sodium Hypochlorite (NaOCl) is used for its antibacterial and tissue-dissolving properties in eliminating root canal infection. It should be continuously replen

3/5 cevaplanıyor: What are the clinical signs of irreversible pulpitis?
✅ 6.8sn | Irreversible pulpitis presents as an emergency condition with persistent symptoms requiring immediate treatment. It may arise from deep caries or tooth structure loss in asymptomatic form. Confusion c

4/5 cevaplanıyor: How should a clinician manage a separated instrument in the root canal?
✅ 6.5sn | A radiograph should be taken immediately to confirm the separation's location and size, advising the patient of the unc

,Soru_No,Soru,Asistan_Cevabi,Sure_sn
0,1,What is the main purpose of EDTA in endodontics?,EDTA is used as a canal irrigant in endodontic...,11.28
1,2,What are the primary objectives of using Sodiu...,Sodium Hypochlorite (NaOCl) is used for its an...,13.73
2,3,What are the clinical signs of irreversible pu...,Irreversible pulpitis presents as an emergency...,6.77
3,4,How should a clinician manage a separated inst...,A radiograph should be taken immediately to co...,6.51
4,5,Explain the importance of working length deter...,Working length determination is crucial in end...,6.83


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


metinler = [c["text"] for c in chunks]
sayfalar = [c["page"] for c in chunks]

vect      = TfidfVectorizer(stop_words="english")
tfidf_mat = vect.fit_transform(metinler)

gold_liste = []
for soru in TEST_SORULARI:
    soru_vec   = vect.transform([soru])
    skorlar    = cosine_similarity(soru_vec, tfidf_mat)[0]
    top5_idx   = skorlar.argsort()[::-1][:5]
    top5_pages = list(dict.fromkeys([sayfalar[j] for j in top5_idx]))
    gold_liste.append("|".join(map(str, top5_pages)))

df_eval = pd.DataFrame({
    "question":   TEST_SORULARI,
    "gold_pages": gold_liste
})

def parse_gold_pages(value):
    return set(int(x.strip()) for x in str(value).split("|") if x.strip().isdigit())

def get_retrieved_pages(question, k=K_RETRIEVE):
    docs  = similarity_search(question, k=k)
    pages = [d["metadata"].get("page") for d in docs if d["metadata"].get("page") not in (None, "?")]
    seen  = set()
    return [p for p in pages if not (p in seen or seen.add(p))]

def precision_at_k(pages, gold, k):
    return sum(1 for p in pages[:k] if p in gold) / k if k else 0.0

def recall_at_k(pages, gold, k):
    return 1.0 if set(pages[:k]) & gold else 0.0

def mrr(pages, gold):
    for rank, page in enumerate(pages, 1):
        if page in gold:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(pages, gold, k):
    dcg  = sum((2 ** int(p in gold) - 1) / math.log2(i + 1) for i, p in enumerate(pages[:k], 1))
    idcg = 1.0 if gold else 0.0
    return dcg / idcg if idcg else 0.0

retrieval_rows = []
for _, row in df_eval.iterrows():
    pages = get_retrieved_pages(row["question"])
    gold  = parse_gold_pages(row["gold_pages"])
    retrieval_rows.append({
        "question":        row["question"],
        "gold_pages":      sorted(gold),
        "retrieved_pages": pages,
        "precision_at_5":  precision_at_k(pages, gold, 5),
        "recall_at_5":     recall_at_k(pages, gold, 5),
        "mrr":             mrr(pages, gold),
        "ndcg_at_5":       ndcg_at_k(pages, gold, 5),
    })

df_retrieval = pd.DataFrame(retrieval_rows)

print("Retrieval metrikleri:")
print(df_retrieval[["question","precision_at_5","recall_at_5","mrr","ndcg_at_5"]].to_string())
print(f"\nOrtalama Precision@5 : {df_retrieval['precision_at_5'].mean():.3f}")
print(f"Ortalama Recall@5    : {df_retrieval['recall_at_5'].mean():.3f}")
print(f"Ortalama MRR         : {df_retrieval['mrr'].mean():.3f}")
print(f"Ortalama nDCG@5      : {df_retrieval['ndcg_at_5'].mean():.3f}")

df_retrieval

Retrieval metrikleri:
                                                                  question  precision_at_5  recall_at_5       mrr  ndcg_at_5
0                         What is the main purpose of EDTA in endodontics?             0.4          1.0  1.000000   1.430677
1    What are the primary objectives of using Sodium Hypochlorite (NaOCl)?             0.2          1.0  1.000000   1.000000
2                    What are the clinical signs of irreversible pulpitis?             0.8          1.0  1.000000   2.448459
3  How should a clinician manage a separated instrument in the root canal?             0.2          1.0  0.333333   0.500000
4   Explain the importance of working length determination in endodontics.             0.2          1.0  1.000000   1.000000

Ortalama Precision@5 : 0.360
Ortalama Recall@5    : 1.000
Ortalama MRR         : 0.867
Ortalama nDCG@5      : 1.276


,question,gold_pages,retrieved_pages,precision_at_5,recall_at_5,mrr,ndcg_at_5
0,What is the main purpose of EDTA in endodontics?,"[967, 968, 1087, 1456, 2346]","[1456, 1457, 985, 1087, 1603]",0.4,1.0,1.000000,1.430677
1,What are the primary objectives of using Sodiu...,"[950, 953, 956, 1628, 2189]","[956, 3110, 2198, 952, 2832]",0.2,1.0,1.000000,1.000000
2,What are the clinical signs of irreversible pu...,"[252, 1845, 2369, 2371, 2864]","[2369, 252, 1846, 1845, 2371]",0.8,1.0,1.000000,2.448459
3,How should a clinician manage a separated inst...,"[140, 2430, 2434, 2448, 2454]","[1290, 1291, 2454, 2431, 2428]",0.2,1.0,0.333333,0.500000
4,Explain the importance of working length deter...,"[1120, 1151, 1249, 2426, 3451]","[1249, 3245, 924, 29, 259]",0.2,1.0,1.000000,1.000000


In [ ]:
import numpy as np
print("numpy:", np.__version__)

numpy: 1.26.4


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

KLINIK_SOZLUK = {
    0: ["edta", "smear", "layer", "demineralize", "chelating", "inorganic",
        "calcium", "dentin", "residual", "obturating", "canal", "removing"],
    1: ["naocl", "hypochlorite", "disinfect", "dissolution", "antimicrobial",
        "organic", "tissue", "irrigat", "sodium", "soft", "debris", "calcified"],
    2: ["irreversible", "pulpitis", "pain", "thermal", "lingering", "spontaneous",
        "pulp", "cold", "stimulus", "endodontic", "sensation", "intensif"],
    3: ["separated", "instrument", "bypass", "ultrasonic", "retrieval", "file",
        "fragment", "removal", "canal", "mechanic", "technique"],
    4: ["working", "length", "cleaning", "shaping", "root", "canal", "system",
        "apical", "overextent", "fracture", "underprepared", "crucial"],
}

REFERANSLAR = [
    "EDTA is used for removing residual obturating materials from the canal after rotary gutta-percha removal.",
    "The primary objectives of using NaOCl include disinfection, dissolution of soft tissue remnants, and removal of calcified debris.",
    "Clinical signs of irreversible pulpitis include lack of response to cold stimuli, lingering or intensifying pain after stimulus removal.",
    "Management of a separated instrument involves ultrasonics or mechanical techniques for removal or bypass within the root canal.",
    "Working length determination is crucial to ensure complete canal cleaning and shaping, preventing overextension and root fractures.",
]

def temizle(text):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", str(text).lower())).strip()

def keyword_capture(answer, keywords):
    cleaned = temizle(answer)
    found   = [kw for kw in keywords if temizle(kw) in cleaned]
    missing = [kw for kw in keywords if kw not in found]
    score   = round(len(found) / len(keywords), 3) if keywords else 0.0
    return score, found, missing

kw_rows = []
for i, row in df_final.iterrows():
    score, found, missing = keyword_capture(row["Asistan_Cevabi"], KLINIK_SOZLUK.get(i, []))
    kw_rows.append({
        "Soru_No":          row["Soru_No"],
        "Keyword_Hit_Rate": score,
        "Bulunan":          ", ".join(found),
        "Eksik":            ", ".join(missing),
    })

df_kw = pd.DataFrame(kw_rows)

sem_model = SentenceTransformer(EMBEDDING_MODEL, device=embedding_device)
sim_rows  = []
for i, ref in enumerate(REFERANSLAR):
    pred  = df_final.iloc[i]["Asistan_Cevabi"]
    vect  = TfidfVectorizer()
    mat   = vect.fit_transform([temizle(ref), temizle(pred)])
    tfidf = cosine_similarity(mat[0:1], mat[1:2])[0][0]
    vecs  = sem_model.encode([ref, pred], normalize_embeddings=True)
    sem   = float(np.dot(vecs[0], vecs[1]))
    sim_rows.append({
        "Soru_No":         i + 1,
        "TFIDF_Cosine":    round(tfidf, 3),
        "Semantic_Cosine": round(sem,   3),
    })

df_sim     = pd.DataFrame(sim_rows)
df_quality = df_kw.merge(df_sim, on="Soru_No")
df_quality

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

,Soru_No,Keyword_Hit_Rate,Bulunan,Eksik,TFIDF_Cosine,Semantic_Cosine
0,1,0.250,"edta, dentin, canal","smear, layer, demineralize, chelating, inorgan...",0.168,0.803
1,2,0.333,"naocl, hypochlorite, tissue, sodium","disinfect, dissolution, antimicrobial, organic...",0.071,0.721
2,3,0.250,"irreversible, pulpitis, pulp","pain, thermal, lingering, spontaneous, cold, s...",0.066,0.772
3,4,0.000,,"separated, instrument, bypass, ultrasonic, ret...",0.133,0.579
4,5,0.333,"working, length, root, crucial","cleaning, shaping, canal, system, apical, over...",0.356,0.821


In [ ]:
df_summary = pd.DataFrame([{
    "Embedding_Model":       "BAAI/bge-large-en-v1.5",
    "Base_Model":            BASE_MODEL_ID,
    "Mean_Keyword_Hit_Rate": df_quality["Keyword_Hit_Rate"].mean(),
    "Mean_TFIDF_Cosine":     df_quality["TFIDF_Cosine"].mean(),
    "Mean_Semantic_Cosine":  df_quality["Semantic_Cosine"].mean(),
    "Mean_Precision_at_5":   df_retrieval["precision_at_5"].mean(),
    "Mean_Recall_at_5":      df_retrieval["recall_at_5"].mean(),
    "Mean_MRR":              df_retrieval["mrr"].mean(),
    "Mean_NDCG_at_5":        df_retrieval["ndcg_at_5"].mean(),
}])

print("\n" + "="*55)
print("SONUÇ ÖZETİ")
print("="*55)
for col in df_summary.columns[2:]:
    print(f"  {col:<25}: {df_summary[col].values[0]:.3f}")
print("="*55)

df_final.to_csv(      f"{OUTPUT_DIR}/colab_hybrid_cevaplar.csv",          index=False)
df_retrieval.to_csv(  f"{OUTPUT_DIR}/colab_hybrid_retrieval_metrics.csv",  index=False)
df_quality.to_csv(    f"{OUTPUT_DIR}/colab_hybrid_quality_metrics.csv",    index=False)
df_summary.to_csv(    f"{OUTPUT_DIR}/colab_hybrid_ozet.csv",               index=False)

print("✅ Kaydedildi: Drive/endodonti_rag/outputs/")
df_summary


SONUÇ ÖZETİ
  Mean_Keyword_Hit_Rate    : 0.233
  Mean_TFIDF_Cosine        : 0.159
  Mean_Semantic_Cosine     : 0.739
  Mean_Precision_at_5      : 0.360
  Mean_Recall_at_5         : 1.000
  Mean_MRR                 : 0.867
  Mean_NDCG_at_5           : 1.276
✅ Kaydedildi: Drive/endodonti_rag/outputs/


,Embedding_Model,Base_Model,Mean_Keyword_Hit_Rate,Mean_TFIDF_Cosine,Mean_Semantic_Cosine,Mean_Precision_at_5,Mean_Recall_at_5,Mean_MRR,Mean_NDCG_at_5
0,BAAI/bge-large-en-v1.5,mistralai/Mistral-7B-Instruct-v0.2,0.2332,0.1588,0.7392,0.36,1.0,0.866667,1.275827


In [ ]:
for i, row in df_quality.iterrows():
    print(f"Soru {i+1} | KW: {row['Keyword_Hit_Rate']} | Bulunan: {row['Bulunan']} | Eksik: {row['Eksik']}")

Soru 1 | KW: 0.25 | Bulunan: edta, dentin, canal | Eksik: smear, layer, demineralize, chelating, inorganic, calcium, residual, obturating, removing
Soru 2 | KW: 0.333 | Bulunan: naocl, hypochlorite, tissue, sodium | Eksik: disinfect, dissolution, antimicrobial, organic, irrigat, soft, debris, calcified
Soru 3 | KW: 0.25 | Bulunan: irreversible, pulpitis, pulp | Eksik: pain, thermal, lingering, spontaneous, cold, stimulus, endodontic, sensation, intensif
Soru 4 | KW: 0.0 | Bulunan:  | Eksik: separated, instrument, bypass, ultrasonic, retrieval, file, fragment, removal, canal, mechanic, technique
Soru 5 | KW: 0.333 | Bulunan: working, length, root, crucial | Eksik: cleaning, shaping, canal, system, apical, overextent, fracture, underprepared


In [ ]:
!pip install -q rouge-score nltk

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

True

In [ ]:
scorer_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
smooth       = SmoothingFunction().method1

nlp_rows = []

for i, (ref, pred) in enumerate(zip(REFERANSLAR, df_final["Asistan_Cevabi"])):
    ref_temiz  = temizle(ref)
    pred_temiz = temizle(pred)
    ref_token  = ref_temiz.split()
    pred_token = pred_temiz.split()

    # BLEU — precision odaklı, n-gram örtüşmesi
    bleu = sentence_bleu([ref_token], pred_token, smoothing_function=smooth)

    # ROUGE-1, ROUGE-2, ROUGE-L — precision + recall + F1
    rouge = scorer_rouge.score(ref_temiz, pred_temiz)

    # METEOR — precision + recall + hizalama
    meteor = meteor_score([ref_token], pred_token)

    nlp_rows.append({
        "Soru_No":  i + 1,
        "BLEU":     round(bleu,                        3),
        "ROUGE-1":  round(rouge["rouge1"].fmeasure,    3),
        "ROUGE-2":  round(rouge["rouge2"].fmeasure,    3),
        "ROUGE-L":  round(rouge["rougeL"].fmeasure,    3),
        "METEOR":   round(meteor,                      3),
        "R1_P":     round(rouge["rouge1"].precision,   3),   # precision
        "R1_R":     round(rouge["rouge1"].recall,      3),   # recall
    })

df_nlp = pd.DataFrame(nlp_rows)

print("\n" + "="*60)
print("NLP METRİK ÖZETİ")
print("="*60)
for col in ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR", "R1_P", "R1_R"]:
    print(f"  {col:<12}: {df_nlp[col].mean():.3f}")
print("="*60)

df_nlp


NLP METRİK ÖZETİ
  BLEU        : 0.045
  ROUGE-1     : 0.252
  ROUGE-2     : 0.076
  ROUGE-L     : 0.197
  METEOR      : 0.215
  R1_P        : 0.207
  R1_R        : 0.324


,Soru_No,BLEU,ROUGE-1,ROUGE-2,ROUGE-L,METEOR,R1_P,R1_R
0,1,0.039,0.279,0.098,0.233,0.175,0.222,0.375
1,2,0.011,0.200,0.000,0.100,0.109,0.182,0.222
2,3,0.015,0.125,0.043,0.125,0.128,0.103,0.158
3,4,0.008,0.200,0.000,0.120,0.129,0.156,0.278
4,5,0.150,0.455,0.238,0.409,0.532,0.370,0.588


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

metinler = [c["text"] for c in chunks]
sayfalar = [c["page"] for c in chunks]

vect      = TfidfVectorizer(stop_words="english")
tfidf_mat = vect.fit_transform(metinler)

for i, item in enumerate(SORU_HAVUZU):
    soru_vec   = vect.transform([item["soru"]])
    skorlar    = cosine_similarity(soru_vec, tfidf_mat)[0]
    top5_idx   = skorlar.argsort()[::-1][:5]
    top5_pages = list(dict.fromkeys([sayfalar[j] for j in top5_idx]))
    SORU_HAVUZU[i]["gold_pages"] = "|".join(map(str, top5_pages))

print(f"Gold pages guncellendi. Ornek:")
print(f"  {SORU_HAVUZU[0]['soru'][:55]}")
print(f"  Gold: {SORU_HAVUZU[0]['gold_pages']}")

Gold pages guncellendi. Ornek:
  What is the main purpose of EDTA in endodontics?
  Gold: 968|2346|1456|967|1087


In [ ]:
import numpy as np
from collections import Counter

SORU_HAVUZU = [
    # KONU 1: İRRİGASYON & DEZENFEKSİYON (1-10)
    {"konu": "İrrigasyon & Dezenfeksiyon", "soru": "What is the main purpose of EDTA in endodontics?", "gold_pages": "2869|1252|815|1392|947", "referans": "EDTA is used for removing residual obturating materials and the smear layer from the canal after instrumentation.", "keywords": ["edta", "smear", "layer", "canal", "removing", "chelating", "inorganic"]},
    {"konu": "İrrigasyon & Dezenfeksiyon", "soru": "What are the primary objectives of using Sodium Hypochlorite (NaOCl) in endodontics?", "gold_pages": "781|782|2090|785|799", "referans": "NaOCl dissolves organic tissue and has broad-spectrum antimicrobial activity, making it the gold standard irrigant.", "keywords": ["naocl", "hypochlorite", "disinfect", "dissolution", "antimicrobial", "organic", "tissue"]},
    {"konu": "İrrigasyon & Dezenfeksiyon", "soru": "What are the properties of an ideal root canal irrigant?", "gold_pages": "", "referans": "An ideal irrigant should be antimicrobial, tissue dissolving, non-toxic, biocompatible, and able to remove the smear layer.", "keywords": ["irrigant", "antimicrobial", "dissolv", "toxic", "smear", "layer", "biocompatib"]},
    {"konu": "İrrigasyon & Dezenfeksiyon", "soru": "How does ultrasonic irrigation enhance root canal disinfection?", "gold_pages": "", "referans": "Ultrasonic irrigation activates irrigant through acoustic streaming and cavitation, enhancing debris removal and disinfection.", "keywords": ["ultrasonic", "irrigation", "acoustic", "cavitation", "debris", "disinfection", "streaming"]},
    {"konu": "İrrigasyon & Dezenfeksiyon", "soru": "What is the role of chlorhexidine in root canal irrigation?", "gold_pages": "", "referans": "Chlorhexidine is used as an alternative or adjunct irrigant due to its substantivity and broad-spectrum antimicrobial properties.", "keywords": ["chlorhexidine", "substantivity", "antimicrobial", "irrigant", "adjunct", "canal"]},
    {"konu": "İrrigasyon & Dezenfeksiyon", "soru": "What is the smear layer and why should it be removed during root canal treatment?", "gold_pages": "", "referans": "The smear layer is a thin layer of debris on canal walls that harbors bacteria and prevents proper sealer penetration.", "keywords": ["smear", "layer", "debris", "bacteria", "sealer", "penetration", "canal", "wall"]},
    {"konu": "İrrigasyon & Dezenfeksiyon", "soru": "What are the complications of sodium hypochlorite extrusion beyond the apex?", "gold_pages": "", "referans": "NaOCl extrusion causes severe pain, swelling, tissue necrosis, and paresthesia due to its caustic and cytotoxic nature.", "keywords": ["naocl", "extrusion", "apex", "pain", "swelling", "necrosis", "paresthesia", "cytotoxic"]},
    {"konu": "İrrigasyon & Dezenfeksiyon", "soru": "How does EDTA interact with NaOCl during root canal irrigation?", "gold_pages": "", "referans": "EDTA chelates calcium ions to remove the inorganic smear layer, while NaOCl removes the organic component; they should not be mixed simultaneously.", "keywords": ["edta", "naocl", "chelate", "calcium", "inorganic", "organic", "smear", "irrigat"]},
    {"konu": "İrrigasyon & Dezenfeksiyon", "soru": "What is passive ultrasonic irrigation and when is it used?", "gold_pages": "", "referans": "Passive ultrasonic irrigation uses an ultrasonically activated file without cutting to agitate and distribute irrigant throughout the canal.", "keywords": ["passive", "ultrasonic", "irrigat", "agitat", "file", "canal", "distribut"]},
    {"konu": "İrrigasyon & Dezenfeksiyon", "soru": "Why is saline used as an irrigant in root canal treatment?", "gold_pages": "", "referans": "Saline is used as a biocompatible flushing agent to remove debris and dilute toxic irrigants without causing tissue damage.", "keywords": ["saline", "biocompatib", "flush", "debris", "dilute", "toxic", "tissue"]},

    # KONU 2: TANI & PULPA PATOLOJİSİ (11-20)
    {"konu": "Tanı & Pulpa Patolojisi", "soru": "What are the clinical signs of irreversible pulpitis?", "gold_pages": "218|2449|2050|217", "referans": "Irreversible pulpitis presents with sharp lingering pain to thermal stimuli, spontaneous pain, and may require endodontic treatment.", "keywords": ["irreversible", "pulpitis", "pain", "thermal", "lingering", "spontaneous", "stimulus"]},
    {"konu": "Tanı & Pulpa Patolojisi", "soru": "How is pulp necrosis diagnosed clinically?", "gold_pages": "", "referans": "Pulp necrosis is diagnosed through negative response to vitality tests, periapical radiographic changes, and possible discoloration.", "keywords": ["necrosis", "vitality", "radiograph", "diagnosis", "periapical", "discolorat", "negative"]},
    {"konu": "Tanı & Pulpa Patolojisi", "soru": "What is the difference between reversible and irreversible pulpitis?", "gold_pages": "", "referans": "Reversible pulpitis causes brief pain that subsides, while irreversible pulpitis causes lingering pain requiring pulp removal.", "keywords": ["reversible", "irreversible", "pulpitis", "lingering", "brief", "subside", "pulp", "removal"]},
    {"konu": "Tanı & Pulpa Patolojisi", "soru": "What are the vitality tests used in endodontic diagnosis?", "gold_pages": "", "referans": "Vitality tests include cold test, heat test, and electric pulp test to assess the sensory response of the pulp tissue.", "keywords": ["vitality", "cold", "heat", "electric", "pulp", "test", "sensory", "diagnosis"]},
    {"konu": "Tanı & Pulpa Patolojisi", "soru": "What is symptomatic apical periodontitis and how does it present?", "gold_pages": "", "referans": "Symptomatic apical periodontitis presents with pain on biting and palpation due to inflammation at the periapical tissues.", "keywords": ["symptomatic", "apical", "periodontitis", "pain", "biting", "palpation", "periapical", "inflammat"]},
    {"konu": "Tanı & Pulpa Patolojisi", "soru": "How does a periapical abscess differ from a periodontal abscess?", "gold_pages": "", "referans": "A periapical abscess originates from pulp necrosis, while a periodontal abscess arises from periodontal pockets without pulp involvement.", "keywords": ["periapical", "periodontal", "abscess", "necrosis", "pocket", "pulp", "origin"]},
    {"konu": "Tanı & Pulpa Patolojisi", "soru": "What is internal resorption and how is it diagnosed?", "gold_pages": "", "referans": "Internal resorption is dentin destruction from within the canal by odontoclasts, diagnosed radiographically as a symmetric oval radiolucency.", "keywords": ["internal", "resorption", "dentin", "odontoclast", "radiograph", "radiolucency", "canal"]},
    {"konu": "Tanı & Pulpa Patolojisi", "soru": "What is the significance of sinus tract in endodontic diagnosis?", "gold_pages": "", "referans": "A sinus tract indicates chronic periapical infection and its tracing with gutta-percha helps identify the source tooth.", "keywords": ["sinus", "tract", "chronic", "periapical", "infection", "gutta", "percha", "tracing"]},
    {"konu": "Tanı & Pulpa Patolojisi", "soru": "How does diabetes affect pulpal and periapical tissues?", "gold_pages": "", "referans": "Diabetes impairs immune response and microvascular circulation, leading to increased susceptibility to pulpal necrosis and periapical pathology.", "keywords": ["diabetes", "immune", "microvascular", "necrosis", "periapical", "pathology", "susceptib"]},
    {"konu": "Tanı & Pulpa Patolojisi", "soru": "What is the role of cone beam CT in endodontic diagnosis?", "gold_pages": "", "referans": "CBCT provides three-dimensional imaging for complex canal anatomy, vertical root fractures, resorptions, and periapical lesions.", "keywords": ["cbct", "cone", "beam", "three", "dimensional", "anatomy", "fracture", "resorption", "periapical"]},

    # KONU 3: KANAL ŞEKİLLENDİRME & ALET (21-30)
    {"konu": "Kanal Şekillendirme & Alet", "soru": "Explain the importance of working length determination in endodontics.", "gold_pages": "2688|2827|247|543|541", "referans": "Working length determination ensures instrumentation stays within the root canal system, preventing apical damage and overextension.", "keywords": ["working", "length", "cleaning", "shaping", "root", "canal", "apical", "overextension"]},
    {"konu": "Kanal Şekillendirme & Alet", "soru": "How should a clinician manage a separated instrument in the root canal?", "gold_pages": "2111|1094|2105|2106|1897", "referans": "Management of a separated instrument involves bypass attempts, ultrasonic retrieval, or cleaning and shaping to the level of the file.", "keywords": ["separated", "instrument", "bypass", "ultrasonic", "retrieval", "file", "removal", "canal"]},
    {"konu": "Kanal Şekillendirme & Alet", "soru": "How does electronic apex locator work in determining working length?", "gold_pages": "", "referans": "Electronic apex locators measure impedance differences between oral mucosa and the apical foramen to accurately determine working length.", "keywords": ["apex", "locator", "impedance", "foramen", "working", "length", "electronic", "measure"]},
    {"konu": "Kanal Şekillendirme & Alet", "soru": "What are the advantages of nickel-titanium rotary instruments over stainless steel files?", "gold_pages": "", "referans": "Nickel-titanium rotary instruments offer superior flexibility and resistance to cyclic fatigue, reducing canal transportation and procedural errors.", "keywords": ["nickel", "titanium", "rotary", "stainless", "flexibility", "fatigue", "transportation", "canal"]},
    {"konu": "Kanal Şekillendirme & Alet", "soru": "What is the crown-down technique in root canal preparation?", "gold_pages": "", "referans": "The crown-down technique prepares the canal from the coronal third downward, reducing coronal interference and improving irrigant penetration.", "keywords": ["crown", "down", "technique", "coronal", "irrigant", "penetration", "preparation", "canal"]},
    {"konu": "Kanal Şekillendirme & Alet", "soru": "What is canal transportation and how can it be prevented?", "gold_pages": "", "referans": "Canal transportation is deviation of the prepared canal from its original path, prevented by using flexible NiTi instruments and proper technique.", "keywords": ["transportation", "canal", "deviation", "flexible", "niti", "instrument", "prevention", "path"]},
    {"konu": "Kanal Şekillendirme & Alet", "soru": "What is the significance of patency filing in root canal treatment?", "gold_pages": "", "referans": "Patency filing keeps the apical foramen clear of debris by passing a small file through it without widening the apical constriction.", "keywords": ["patency", "filing", "apical", "foramen", "debris", "constriction", "small", "file"]},
    {"konu": "Kanal Şekillendirme & Alet", "soru": "What is cyclic fatigue in endodontic instruments and how is it managed?", "gold_pages": "", "referans": "Cyclic fatigue occurs due to repeated flexion and stress of rotating instruments in curved canals, managed by single-use protocols.", "keywords": ["cyclic", "fatigue", "flexion", "stress", "rotating", "curved", "canal", "single", "use"]},
    {"konu": "Kanal Şekillendirme & Alet", "soru": "What are reciprocating systems in endodontic instrumentation?", "gold_pages": "", "referans": "Reciprocating systems use a single file in a back-and-forth motion, reducing torsional stress and cyclic fatigue compared to continuous rotation.", "keywords": ["reciprocat", "single", "file", "torsional", "stress", "cyclic", "fatigue", "rotation", "motion"]},
    {"konu": "Kanal Şekillendirme & Alet", "soru": "What is the apical constriction and why is it the ideal endpoint of preparation?", "gold_pages": "", "referans": "The apical constriction is the narrowest point of the root canal and serves as the ideal endpoint for preparation and obturation.", "keywords": ["apical", "constriction", "narrow", "endpoint", "preparation", "obturation", "canal"]},

    # KONU 4: OBTÜRASYON & DOLUM (31-40)
    {"konu": "Obtürasyon & Dolum", "soru": "How is gutta-percha used in root canal obturation?", "gold_pages": "", "referans": "Gutta-percha is the standard root canal filling material used to achieve a hermetic three-dimensional seal of the canal system.", "keywords": ["gutta", "percha", "obturation", "filling", "seal", "canal", "hermetic", "three", "dimensional"]},
    {"konu": "Obtürasyon & Dolum", "soru": "What is the role of MTA in endodontic treatment?", "gold_pages": "", "referans": "MTA is used for perforation repair, apexification, pulp capping, and root-end filling due to its biocompatibility and sealing ability.", "keywords": ["mta", "perforation", "apexification", "pulp", "capping", "biocompatib", "sealing", "root"]},
    {"konu": "Obtürasyon & Dolum", "soru": "Describe the lateral condensation technique for root canal obturation.", "gold_pages": "", "referans": "Lateral condensation uses a master gutta-percha cone and accessory points compacted with spreaders to fill the canal in three dimensions.", "keywords": ["lateral", "condensation", "master", "cone", "accessory", "spreader", "gutta", "percha", "canal"]},
    {"konu": "Obtürasyon & Dolum", "soru": "What is warm vertical compaction and what are its advantages?", "gold_pages": "", "referans": "Warm vertical compaction uses heat-softened gutta-percha compacted vertically to achieve superior adaptation and a hermetic seal.", "keywords": ["warm", "vertical", "compaction", "heat", "gutta", "percha", "adaptation", "hermetic", "seal"]},
    {"konu": "Obtürasyon & Dolum", "soru": "What are the functions of root canal sealers?", "gold_pages": "", "referans": "Root canal sealers fill the space between the gutta-percha and canal walls, preventing recontamination and improving the seal quality.", "keywords": ["sealer", "gutta", "percha", "canal", "wall", "recontamination", "seal", "space", "filling"]},
    {"konu": "Obtürasyon & Dolum", "soru": "What is the single-cone obturation technique?", "gold_pages": "", "referans": "Single-cone obturation uses one gutta-percha cone matched to the final instrument size with a bioceramic sealer to fill the canal.", "keywords": ["single", "cone", "obturation", "gutta", "percha", "bioceramic", "sealer", "canal", "instrument"]},
    {"konu": "Obtürasyon & Dolum", "soru": "What are the properties of an ideal root canal filling material?", "gold_pages": "", "referans": "An ideal filling material should be dimensionally stable, biocompatible, radiopaque, antibacterial, and provide a hermetic seal.", "keywords": ["filling", "material", "dimensional", "stable", "biocompatib", "radiopaque", "antibacterial", "hermetic"]},
    {"konu": "Obtürasyon & Dolum", "soru": "What is the biological basis of pulp capping treatment?", "gold_pages": "", "referans": "Pulp capping relies on the pulp's regenerative capacity where biocompatible materials stimulate reparative dentin bridge formation.", "keywords": ["pulp", "capping", "regenerative", "reparative", "dentin", "bridge", "biocompatib", "formation"]},
    {"konu": "Obtürasyon & Dolum", "soru": "What is apexification and when is it indicated?", "gold_pages": "", "referans": "Apexification induces apical barrier formation in immature teeth with necrotic pulps using calcium hydroxide or MTA.", "keywords": ["apexification", "apical", "barrier", "immature", "necrotic", "calcium", "hydroxide", "mta"]},
    {"konu": "Obtürasyon & Dolum", "soru": "How does bioceramic sealer differ from traditional epoxy resin sealers?", "gold_pages": "", "referans": "Bioceramic sealers are biocompatible, dimensionally stable, and hydrophilic, whereas epoxy resin sealers may shrink and be cytotoxic.", "keywords": ["bioceramic", "sealer", "epoxy", "resin", "biocompatib", "dimensional", "hydrophilic", "cytotoxic"]},

    # KONU 5: KOMPLİKASYONLAR & BAŞARISIZLIK (41-50)
    {"konu": "Komplikasyonlar & Başarısızlık", "soru": "What are the main causes of endodontic treatment failure?", "gold_pages": "", "referans": "Endodontic failure results from inadequate debridement, missed canals, coronal leakage, or persistent microbial infection.", "keywords": ["failure", "debridement", "missed", "canal", "leakage", "microbial", "infection", "persistent"]},
    {"konu": "Komplikasyonlar & Başarısızlık", "soru": "What are the indications for root canal retreatment?", "gold_pages": "", "referans": "Retreatment is indicated when primary treatment has failed, with persistent symptoms or radiographic evidence of periapical pathology.", "keywords": ["retreatment", "failure", "persistent", "symptom", "radiograph", "periapical", "pathology", "canal"]},
    {"konu": "Komplikasyonlar & Başarısızlık", "soru": "What is a vertical root fracture and how is it diagnosed?", "gold_pages": "", "referans": "Vertical root fractures run longitudinally along the root and are diagnosed with CBCT, selective periodontal probing, and clinical symptoms.", "keywords": ["vertical", "root", "fracture", "longitudinal", "cbct", "periodontal", "probing", "diagnosis"]},
    {"konu": "Komplikasyonlar & Başarısızlık", "soru": "How is a root perforation managed in endodontics?", "gold_pages": "", "referans": "Root perforation is managed by immediate sealing with biocompatible materials such as MTA to prevent bacterial contamination.", "keywords": ["perforation", "sealing", "mta", "biocompatib", "bacterial", "contamination", "root", "immediate"]},
    {"konu": "Komplikasyonlar & Başarısızlık", "soru": "What is a flare-up in endodontics and how is it managed?", "gold_pages": "", "referans": "A flare-up is an acute exacerbation of pain and swelling after endodontic treatment, managed with drainage, medications, and reassurance.", "keywords": ["flare", "up", "acute", "pain", "swelling", "drainage", "medication", "exacerbation"]},
    {"konu": "Komplikasyonlar & Başarısızlık", "soru": "What are the causes and management of post-operative pain after root canal treatment?", "gold_pages": "", "referans": "Post-operative pain is caused by periapical inflammation, overextension, or missed canals, managed with analgesics and anti-inflammatories.", "keywords": ["post", "operative", "pain", "periapical", "inflammat", "overextension", "analgesic", "canal"]},
    {"konu": "Komplikasyonlar & Başarısızlık", "soru": "What is ledge formation in root canal treatment and how is it prevented?", "gold_pages": "", "referans": "Ledge formation is an artificial step created in the canal wall due to improper instrumentation, prevented by using precurved files and copious irrigation.", "keywords": ["ledge", "formation", "artificial", "canal", "wall", "instrument", "precurved", "file", "irrigat"]},
    {"konu": "Komplikasyonlar & Başarısızlık", "soru": "What is the impact of coronal leakage on endodontic treatment outcomes?", "gold_pages": "", "referans": "Coronal leakage allows oral bacteria to recontaminate the filled canal, leading to treatment failure and periapical disease recurrence.", "keywords": ["coronal", "leakage", "bacteria", "recontaminate", "canal", "failure", "periapical", "recurrence"]},
    {"konu": "Komplikasyonlar & Başarısızlık", "soru": "How is external root resorption diagnosed and managed endodontically?", "gold_pages": "", "referans": "External resorption is diagnosed radiographically with an irregular radiolucency on the root surface and managed by treating the underlying cause.", "keywords": ["external", "resorption", "radiograph", "radiolucency", "root", "surface", "diagnosis", "cause"]},
    {"konu": "Komplikasyonlar & Başarısızlık", "soru": "What microorganisms are most commonly associated with persistent endodontic infections?", "gold_pages": "", "referans": "Enterococcus faecalis is the most common organism in persistent infections due to its resistance to calcium hydroxide and harsh environments.", "keywords": ["enterococcus", "faecalis", "persistent", "infection", "resistance", "calcium", "hydroxide", "canal"]},
]

print(f"✅ Toplam soru: {len(SORU_HAVUZU)}")
konular = Counter(s["konu"] for s in SORU_HAVUZU)
for konu, sayi in konular.items():
    print(f"  {konu}: {sayi} soru")

✅ Toplam soru: 50
  İrrigasyon & Dezenfeksiyon: 10 soru
  Tanı & Pulpa Patolojisi: 10 soru
  Kanal Şekillendirme & Alet: 10 soru
  Obtürasyon & Dolum: 10 soru
  Komplikasyonlar & Başarısızlık: 10 soru


In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

scorer_rouge_50 = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
smooth_50       = SmoothingFunction().method1

def parse_gold(value):
    return set(int(x.strip()) for x in str(value).split("|") if x.strip().isdigit())

def prec_at_k(pages, gold, k):
    return sum(1 for p in pages[:k] if p in gold) / k if k else 0.0

def rec_at_k(pages, gold, k):
    return 1.0 if set(pages[:k]) & gold else 0.0

def mrr_fn(pages, gold):
    for rank, p in enumerate(pages, 1):
        if p in gold: return 1.0 / rank
    return 0.0

def ndcg_fn(pages, gold, k):
    dcg = sum((2**int(p in gold)-1)/math.log2(i+1) for i, p in enumerate(pages[:k], 1))
    return dcg / 1.0 if gold else 0.0

def kw_hit(cevap, keywords):
    t = temizle(cevap)
    return round(len([k for k in keywords if temizle(k) in t]) / len(keywords), 3) if keywords else 0.0

def nlp_metrik(ref, pred):
    r, p   = temizle(ref).split(), temizle(pred).split()
    bleu   = sentence_bleu([r], p, smoothing_function=smooth_50)
    rouge  = scorer_rouge_50.score(temizle(ref), temizle(pred))
    meteor = meteor_score([r], p)
    vect   = TfidfVectorizer()
    mat    = vect.fit_transform([temizle(ref), temizle(pred)])
    tfidf  = cosine_similarity(mat[0:1], mat[1:2])[0][0]
    vecs   = sem_model.encode([ref, pred], normalize_embeddings=True)
    sem    = float(np.dot(vecs[0], vecs[1]))
    return {
        "BLEU":    round(bleu, 3),
        "ROUGE_1": round(rouge["rouge1"].fmeasure, 3),
        "ROUGE_2": round(rouge["rouge2"].fmeasure, 3),
        "ROUGE_L": round(rouge["rougeL"].fmeasure, 3),
        "METEOR":  round(meteor, 3),
        "R1_P":    round(rouge["rouge1"].precision, 3),
        "R1_R":    round(rouge["rouge1"].recall, 3),
        "TFIDF":   round(tfidf, 3),
        "Semantic":round(sem, 3),
    }

def get_pages(soru, k=5):
    docs  = similarity_search(soru, k=k)
    pages = [d["metadata"].get("page") for d in docs if d["metadata"].get("page") not in (None, "?")]
    seen  = set()
    return [p for p in pages if not (p in seen or seen.add(p))]

In [ ]:
print("\n" + "=" * 70)
print("50 SORULUK TEST BAŞLIYOR")
print("=" * 70)

birikimler = {
    "Keyword_Hit_Rate": [], "TFIDF": [], "Semantic": [],
    "Precision_at_5": [], "Recall_at_5": [], "MRR": [], "NDCG_at_5": [],
    "BLEU": [], "ROUGE_1": [], "ROUGE_2": [], "ROUGE_L": [],
    "METEOR": [], "R1_P": [], "R1_R": [], "Sure_sn": []
}

tum_satirlar = []

for i, item in enumerate(SORU_HAVUZU):
    t0 = time.time()
    try:
        cevap = endodonti_asistanina_sor(item["soru"])
    except Exception as e:
        cevap = f"[HATA] {e}"
    sure = round(time.time() - t0, 2)

    pages = get_pages(item["soru"])
    gold  = parse_gold(item["gold_pages"])
    nlp   = nlp_metrik(item["referans"], cevap)
    kw    = kw_hit(cevap, item["keywords"])
    prec  = prec_at_k(pages, gold, 5)
    rec   = rec_at_k(pages, gold, 5)
    mrr   = mrr_fn(pages, gold)
    ndcg  = ndcg_fn(pages, gold, 5)

    for m, v in [
        ("Keyword_Hit_Rate", kw), ("TFIDF", nlp["TFIDF"]), ("Semantic", nlp["Semantic"]),
        ("Precision_at_5", prec), ("Recall_at_5", rec), ("MRR", mrr), ("NDCG_at_5", ndcg),
        ("BLEU", nlp["BLEU"]), ("ROUGE_1", nlp["ROUGE_1"]), ("ROUGE_2", nlp["ROUGE_2"]),
        ("ROUGE_L", nlp["ROUGE_L"]), ("METEOR", nlp["METEOR"]),
        ("R1_P", nlp["R1_P"]), ("R1_R", nlp["R1_R"]), ("Sure_sn", sure)
    ]:
        birikimler[m].append(v)

    tum_satirlar.append({
        "Soru_No": i+1, "Konu": item["konu"], "Soru": item["soru"],
        "Cevap": cevap[:300], "Sure_sn": sure,
        "Keyword_Hit_Rate": kw, "Precision_at_5": prec,
        "Recall_at_5": rec, "MRR": mrr, "NDCG_at_5": ndcg, **nlp
    })

    print(f"  {i+1:2}/50 [{item['konu'][:20]:<20}] "
          f"⏱{sure}s | KW:{kw:.2f} | RL:{nlp['ROUGE_L']:.2f} | Sem:{nlp['Semantic']:.2f}")


50 SORULUK TEST BAŞLIYOR
   1/50 [İrrigasyon & Dezenfe] ⏱10.98s | KW:0.29 | RL:0.23 | Sem:0.80
   2/50 [İrrigasyon & Dezenfe] ⏱18.16s | KW:0.43 | RL:0.22 | Sem:0.69
   3/50 [İrrigasyon & Dezenfe] ⏱11.69s | KW:0.29 | RL:0.33 | Sem:0.80
   4/50 [İrrigasyon & Dezenfe] ⏱12.37s | KW:0.29 | RL:0.20 | Sem:0.81
   5/50 [İrrigasyon & Dezenfe] ⏱15.81s | KW:0.67 | RL:0.31 | Sem:0.76
   6/50 [İrrigasyon & Dezenfe] ⏱14.06s | KW:0.75 | RL:0.35 | Sem:0.80
   7/50 [İrrigasyon & Dezenfe] ⏱6.77s | KW:0.25 | RL:0.16 | Sem:0.68
   8/50 [İrrigasyon & Dezenfe] ⏱7.04s | KW:0.38 | RL:0.35 | Sem:0.83
   9/50 [İrrigasyon & Dezenfe] ⏱6.69s | KW:0.57 | RL:0.31 | Sem:0.80
  10/50 [İrrigasyon & Dezenfe] ⏱6.98s | KW:0.14 | RL:0.17 | Sem:0.69
  11/50 [Tanı & Pulpa Patoloj] ⏱6.8s | KW:0.29 | RL:0.26 | Sem:0.77
  12/50 [Tanı & Pulpa Patoloj] ⏱8.47s | KW:0.29 | RL:0.20 | Sem:0.81
  13/50 [Tanı & Pulpa Patoloj] ⏱11.67s | KW:0.62 | RL:0.30 | Sem:0.85
  14/50 [Tanı & Pulpa Patoloj] ⏱6.78s | KW:0.50 | RL:0.29 | Sem:0.79
  

In [ ]:
n = len(SORU_HAVUZU)


df_50_detay  = pd.DataFrame(tum_satirlar)

print("\n" + "=" * 70)
print("50 SORU TEST SONUCU")
print("=" * 70)
print("=" * 70)

print("\n📊 NLP METRİK ÖZETİ:")
for m in ["BLEU","ROUGE_1","ROUGE_2","ROUGE_L","METEOR","R1_P","R1_R"]:
    print(f"  {m:<12}: {sum(birikimler[m])/n:.3f}")

print(f"\n⏱ Toplam süre : {sum(birikimler['Sure_sn']):.0f}s")
print(f"⏱ Ort. süre   : {sum(birikimler['Sure_sn'])/n:.1f}s/soru")

print("\n📋 KONU BAZLI ORTALAMA METRİKLER:")
konu_ozet_50 = df_50_detay.groupby("Konu")[
    ["BLEU","ROUGE_L","METEOR","Semantic","Keyword_Hit_Rate","Precision_at_5","MRR"]
].mean().round(3)
print(konu_ozet_50.to_string())

df_50_detay.to_csv( f"{OUTPUT_DIR}/test_50_detay.csv",  index=False)
konu_ozet_50.to_csv(f"{OUTPUT_DIR}/test_50_konu.csv")

print("\n✅ Kaydedilen dosyalar:")
print(f"   → test_50_detay.csv  ({len(df_50_detay)} satır)")
print(f"   → test_50_ozet.csv")



50 SORU TEST SONUCU

📊 NLP METRİK ÖZETİ:
  BLEU        : 0.045
  ROUGE_1     : 0.321
  ROUGE_2     : 0.107
  ROUGE_L     : 0.262
  METEOR      : 0.284
  R1_P        : 0.275
  R1_R        : 0.391

⏱ Toplam süre : 403s
⏱ Ort. süre   : 8.1s/soru

📋 KONU BAZLI ORTALAMA METRİKLER:
                                 BLEU  ROUGE_L  METEOR  Semantic  Keyword_Hit_Rate  Precision_at_5    MRR
Konu                                                                                                     
Kanal Şekillendirme & Alet      0.055    0.291   0.344     0.773             0.463            0.32  0.867
Komplikasyonlar & Başarısızlık  0.037    0.248   0.262     0.809             0.431            0.16  0.320
Obtürasyon & Dolum              0.058    0.270   0.327     0.774             0.435            0.18  0.483
Tanı & Pulpa Patolojisi         0.041    0.239   0.253     0.819             0.389            0.26  0.503
İrrigasyon & Dezenfeksiyon      0.034    0.261   0.233     0.766             0.404    

In [ ]:
for i, item in enumerate(SORU_HAVUZU):
    if not item["gold_pages"].strip():
        pages = get_pages(item["soru"], k=5)
        SORU_HAVUZU[i]["gold_pages"] = "|".join(str(p) for p in pages)

print("✅ Gold pages güncellendi.")

✅ Gold pages güncellendi.


In [ ]:
# ── RAG ONLY TEST ────────────────────────────────────────

from transformers import pipeline

print("RAG Only test başlıyor...")


def rag_only_cevap(question: str) -> str:
    docs    = similarity_search(question, k=10)
    context = build_context(docs, max_chars=1500)

    prompt = (
        "### Instruction:\n"
        "Answer as an endodontist using only the given context. "
        "Give a concise answer in 1-2 sentences maximum.\n\n"
        "### Input:\n"
        f"CONTEXT:\n{context}\n\n"
        f"Question: {question}\n\n"
        "### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536)
    inputs = {k: v.to(base_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = base_model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    if "### Response:" in decoded:
        decoded = decoded.split("### Response:", 1)[-1]
    return decoded.strip()


print("=" * 70)
print("RAG ONLY — 50 SORU TEST")
print("=" * 70)

birikimler_rag = {
    "Keyword_Hit_Rate": [], "TFIDF": [], "Semantic": [],
    "Precision_at_5": [], "Recall_at_5": [], "MRR": [], "NDCG_at_5": [],
    "BLEU": [], "ROUGE_1": [], "ROUGE_2": [], "ROUGE_L": [],
    "METEOR": [], "R1_P": [], "R1_R": [], "Sure_sn": []
}

tum_satirlar_rag = []

for i, item in enumerate(SORU_HAVUZU):
    t0 = time.time()
    try:
        cevap = rag_only_cevap(item["soru"])
    except Exception as e:
        cevap = f"[HATA] {e}"
    sure = round(time.time() - t0, 2)

    pages = get_pages(item["soru"])
    gold  = parse_gold(item["gold_pages"])
    nlp   = nlp_metrik(item["referans"], cevap)
    kw    = kw_hit(cevap, item["keywords"])
    prec  = prec_at_k(pages, gold, 5)
    rec   = rec_at_k(pages, gold, 5)
    mrr   = mrr_fn(pages, gold)
    ndcg  = ndcg_fn(pages, gold, 5)

    for m, v in [
        ("Keyword_Hit_Rate", kw), ("TFIDF", nlp["TFIDF"]), ("Semantic", nlp["Semantic"]),
        ("Precision_at_5", prec), ("Recall_at_5", rec), ("MRR", mrr), ("NDCG_at_5", ndcg),
        ("BLEU", nlp["BLEU"]), ("ROUGE_1", nlp["ROUGE_1"]), ("ROUGE_2", nlp["ROUGE_2"]),
        ("ROUGE_L", nlp["ROUGE_L"]), ("METEOR", nlp["METEOR"]),
        ("R1_P", nlp["R1_P"]), ("R1_R", nlp["R1_R"]), ("Sure_sn", sure)
    ]:
        birikimler_rag[m].append(v)

    tum_satirlar_rag.append({
        "Soru_No": i+1, "Konu": item["konu"], "Soru": item["soru"],
        "Cevap": cevap[:300], "Sure_sn": sure,
        "Keyword_Hit_Rate": kw, "Precision_at_5": prec,
        "Recall_at_5": rec, "MRR": mrr, "NDCG_at_5": ndcg, **nlp
    })

    print(f"  {i+1:2}/50 [{item['konu'][:20]:<20}] "
          f"⏱{sure}s | KW:{kw:.2f} | RL:{nlp['ROUGE_L']:.2f} | Sem:{nlp['Semantic']:.2f}")


n = len(SORU_HAVUZU)

df_rag_ozet = pd.DataFrame([{
    "Embedding_Model":       "MultiQA",
    "Base_Model":            "google/flan-t5-base (RAG Only)",
    "Mean_Keyword_Hit_Rate": round(sum(birikimler_rag["Keyword_Hit_Rate"]) / n, 4),
    "Mean_TFIDF_Cosine":     round(sum(birikimler_rag["TFIDF"])            / n, 4),
    "Mean_Semantic_Cosine":  round(sum(birikimler_rag["Semantic"])         / n, 4),
    "Mean_Precision_at_5":   round(sum(birikimler_rag["Precision_at_5"])   / n, 4),
    "Mean_Recall_at_5":      round(sum(birikimler_rag["Recall_at_5"])      / n, 4),
    "Mean_MRR":              round(sum(birikimler_rag["MRR"])              / n, 4),
    "Mean_NDCG_at_5":        round(sum(birikimler_rag["NDCG_at_5"])        / n, 4),
}])

df_rag_detay = pd.DataFrame(tum_satirlar_rag)

print("\n" + "=" * 70)
print("RAG ONLY SONUCU")
print("=" * 70)
print(df_rag_ozet.to_string(index=True))
print("=" * 70)

print("\n📊 NLP METRİK ÖZETİ:")
for m in ["BLEU","ROUGE_1","ROUGE_2","ROUGE_L","METEOR","R1_P","R1_R"]:
    print(f"  {m:<12}: {sum(birikimler_rag[m])/n:.3f}")

print("\n" + "=" * 70)
print("HİBRİT vs RAG KARŞILAŞTIRMA")
print("=" * 70)

karsilastirma = pd.DataFrame([
    {
        "Model": "RAG Only (Flan-T5)",
        "BLEU":     round(sum(birikimler_rag["BLEU"])/n, 3),
        "ROUGE_L":  round(sum(birikimler_rag["ROUGE_L"])/n, 3),
        "METEOR":   round(sum(birikimler_rag["METEOR"])/n, 3),
        "Semantic": round(sum(birikimler_rag["Semantic"])/n, 3),
        "KW_Hit":   round(sum(birikimler_rag["Keyword_Hit_Rate"])/n, 3),
        "Prec@5":   round(sum(birikimler_rag["Precision_at_5"])/n, 3),
        "MRR":      round(sum(birikimler_rag["MRR"])/n, 3),
    },
    {
        "Model": "Hibrit (FT+RAG Mistral-7B)",
        "BLEU":     round(sum(birikimler["BLEU"])/n, 3),
        "ROUGE_L":  round(sum(birikimler["ROUGE_L"])/n, 3),
        "METEOR":   round(sum(birikimler["METEOR"])/n, 3),
        "Semantic": round(sum(birikimler["Semantic"])/n, 3),
        "KW_Hit":   round(sum(birikimler["Keyword_Hit_Rate"])/n, 3),
        "Prec@5":   round(sum(birikimler["Precision_at_5"])/n, 3),
        "MRR":      round(sum(birikimler["MRR"])/n, 3),
    }
])

print(karsilastirma.to_string(index=False))


df_rag_detay.to_csv( f"{OUTPUT_DIR}/rag_only_50_detay.csv",  index=False)
df_rag_ozet.to_csv(  f"{OUTPUT_DIR}/rag_only_50_ozet.csv",   index=False)
karsilastirma.to_csv(f"{OUTPUT_DIR}/hibrit_vs_rag_karsilastirma.csv", index=False)

print("\n✅ Kaydedildi.")
karsilastirma

RAG Only test başlıyor...
RAG ONLY — 50 SORU TEST
   1/50 [İrrigasyon & Dezenfe] ⏱10.24s | KW:0.29 | RL:0.13 | Sem:0.77
   2/50 [İrrigasyon & Dezenfe] ⏱9.62s | KW:0.43 | RL:0.15 | Sem:0.67
   3/50 [İrrigasyon & Dezenfe] ⏱9.08s | KW:0.29 | RL:0.25 | Sem:0.77
   4/50 [İrrigasyon & Dezenfe] ⏱10.38s | KW:0.43 | RL:0.17 | Sem:0.76
   5/50 [İrrigasyon & Dezenfe] ⏱9.93s | KW:0.33 | RL:0.32 | Sem:0.71
   6/50 [İrrigasyon & Dezenfe] ⏱9.37s | KW:0.75 | RL:0.29 | Sem:0.81
   7/50 [İrrigasyon & Dezenfe] ⏱9.9s | KW:0.38 | RL:0.17 | Sem:0.69
   8/50 [İrrigasyon & Dezenfe] ⏱10.02s | KW:0.38 | RL:0.10 | Sem:0.70
   9/50 [İrrigasyon & Dezenfe] ⏱9.94s | KW:0.57 | RL:0.27 | Sem:0.80
  10/50 [İrrigasyon & Dezenfe] ⏱9.09s | KW:0.14 | RL:0.22 | Sem:0.72
  11/50 [Tanı & Pulpa Patoloj] ⏱9.91s | KW:0.29 | RL:0.22 | Sem:0.75
  12/50 [Tanı & Pulpa Patoloj] ⏱9.93s | KW:0.29 | RL:0.14 | Sem:0.79
  13/50 [Tanı & Pulpa Patoloj] ⏱9.57s | KW:0.62 | RL:0.31 | Sem:0.83
  14/50 [Tanı & Pulpa Patoloj] ⏱9.38s | KW:0.50 | R

,Model,BLEU,ROUGE_L,METEOR,Semantic,KW_Hit,Prec@5,MRR
0,RAG Only (Flan-T5),0.032,0.221,0.286,0.771,0.439,0.236,0.566
1,Hibrit (FT+RAG Mistral-7B),0.045,0.262,0.284,0.788,0.424,0.236,0.566


In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio
print("✅ Kurulum tamam.")

✅ Kurulum tamam.


In [ ]:
from pyngrok import ngrok

NGROK_TOKEN = "3DdtUSeMwJkilDQN2VxOVg16ulo_533ErKdn8UB5cLvgo1RrH"
ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok token ayarlandı.")

✅ ngrok token ayarlandı.


In [ ]:
import nest_asyncio
import uvicorn
import threading
import time
import torch

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

nest_asyncio.apply()

In [ ]:
class SoruRequest(BaseModel):
    soru: str
    show_context: Optional[bool] = False

class CevapResponse(BaseModel):
    soru: str
    cevap: str
    sure_sn: float
    retrieved_pages: list

class MetrikResponse(BaseModel):
    bleu: float
    rouge_l: float
    meteor: float
    semantic_cosine: float
    keyword_hit_rate: float

In [ ]:
app = FastAPI(
    title="Endodonti Asistanı API",
    description="RAG + Fine-tuned Mistral-7B tabanlı endodonti klinik asistanı",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [ ]:
@app.get("/")
def root():
    return {
        "durum": "aktif",
        "model": "Mistral-7B-Instruct-v0.2 + LoRA",
        "embedding": "multi-qa-mpnet-base-cos-v1",
        "versiyon": "1.0.0"
    }

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.is_available()}

In [ ]:
@app.post("/sor", response_model=CevapResponse)
def sor(req: SoruRequest):
    if not req.soru.strip():
        raise HTTPException(status_code=400, detail="Soru boş olamaz.")

    t0 = time.time()

    try:
        docs    = similarity_search(req.soru, k=5)
        context = build_context(docs)
        cevap   = endodonti_asistanina_sor(req.soru)
        pages   = [d["metadata"].get("page", "?") for d in docs]
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Model hatası: {str(e)}")

    return CevapResponse(
        soru=req.soru,
        cevap=cevap,
        sure_sn=round(time.time() - t0, 2),
        retrieved_pages=pages
    )

In [ ]:
class BatchRequest(BaseModel):
    sorular: list[str]

@app.post("/batch")
def batch_sor(req: BatchRequest):
    if len(req.sorular) > 10:
        raise HTTPException(status_code=400, detail="En fazla 10 soru gönderilebilir.")

    sonuclar = []
    for soru in req.sorular:
        t0 = time.time()
        try:
            cevap = endodonti_asistanina_sor(soru)
            durum = "ok"
        except Exception as e:
            cevap = f"[HATA] {str(e)}"
            durum = "hata"
        sonuclar.append({
            "soru":    soru,
            "cevap":   cevap,
            "sure_sn": round(time.time() - t0, 2),
            "durum":   durum
        })

    return {"sonuclar": sonuclar, "toplam": len(sonuclar)}

In [ ]:
@app.get("/context")
def get_context(soru: str, k: int = 5):
    if not soru.strip():
        raise HTTPException(status_code=400, detail="Soru boş olamaz.")
    docs = similarity_search(soru, k=k)
    return {
        "soru": soru,
        "belgeler": [
            {
                "sira":   i + 1,
                "sayfa":  d["metadata"].get("page", "?"),
                "skor":   d["metadata"].get("score", 0),
                "metin":  d["page_content"][:500]
            }
            for i, d in enumerate(docs)
        ]
    }

print("✅ FastAPI uygulaması tanımlandı.")

✅ FastAPI uygulaması tanımlandı.


In [ ]:
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

# Arka planda çalıştır
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("✅ Sunucu başlatıldı (port 8000)")

✅ Sunucu başlatıldı (port 8000)


In [ ]:
# Eski tünelleri kapat
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

# Yeni tünel aç
tunnel     = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

print("\n" + "="*55)
print("🚀 ENDODONTİ ASISTANI API HAZIR")
print("="*55)
print(f"📡 Public URL : {PUBLIC_URL}")
print(f"📖 Docs       : {PUBLIC_URL}/docs")
print(f"❤️  Health     : {PUBLIC_URL}/health")
print("="*55)


🚀 ENDODONTİ ASISTANI API HAZIR
📡 Public URL : https://unbeneficial-paris-important.ngrok-free.dev
📖 Docs       : https://unbeneficial-paris-important.ngrok-free.dev/docs
❤️  Health     : https://unbeneficial-paris-important.ngrok-free.dev/health


In [ ]:
import requests

test_url = f"{PUBLIC_URL}/sor"
test_payload = {
    "soru": "What is the main purpose of EDTA in endodontics?"
}

print("API test ediliyor...")
response = requests.post(test_url, json=test_payload)

if response.status_code == 200:
    data = response.json()
    print(f"✅ Başarılı!")
    print(f"Soru  : {data['soru']}")
    print(f"Cevap : {data['cevap'][:300]}")
    print(f"Süre  : {data['sure_sn']} sn")
    print(f"Sayfalar: {data['retrieved_pages']}")
else:
    print(f"❌ Hata: {response.status_code}")
    print(response.text)

API test ediliyor...
❌ Hata: 500
{"detail":"Model hatası: name 'similarity_search' is not defined"}
